# LLaMA 3.1 on SageMaker

This notebook shows how to:

- Deploy a Hugging Face LLaMA 3.1 model on Amazon SageMaker  
- Run a quick test inference  
- **Always tear down the endpoint** to avoid idle GPU charges


## 0) Parameters

- Region defaults to the current AWS session, or `ap-southeast-2` if none is set  
- Endpoint resources (endpoint, config, model) will be **auto-cleaned** in a `finally` block


In [1]:
import os, json, time, boto3, botocore, sagemaker
from sagemaker.huggingface import HuggingFaceModel

REGION = os.environ.get("AWS_REGION") or boto3.Session().region_name or "ap-southeast-2"
ROLE = sagemaker.get_execution_role()  # works on Notebook Instance

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
ENDPOINT_NAME = "llama31-8b-endpoint"
INSTANCE_TYPE = "ml.g5.2xlarge"   # GPU cost, keep usage short!
AUTO_TIMEOUT_MIN = 20

IMAGE_URI = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04"

print("Region:", REGION)
print("Role:", ROLE)
print("Model:", MODEL_ID)
print("Endpoint:", ENDPOINT_NAME)
print("Image URI:", IMAGE_URI)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Region: ap-southeast-2
Role: arn:aws:iam::575935529773:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole
Model: meta-llama/Llama-3.1-8B-Instruct
Endpoint: llama31-8b-endpoint
Image URI: 763104351884.dkr.ecr.ap-southeast-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04


## 1) Clients & helpers (cleanup + wait)
Setup SageMaker clients and define small helper functions to wait for endpoints and clean them up safely.

In [2]:
sm = boto3.client("sagemaker", region_name=REGION)
rt = boto3.client("sagemaker-runtime", region_name=REGION)

def safe_call(fn, **kw):
    try:
        return fn(**kw)
    except botocore.exceptions.ClientError as e:
        code = e.response.get("Error", {}).get("Code")
        if code in {"ValidationException", "ResourceNotFound"}:
            return None
        raise

def kill(name: str):
    """Best-effort, idempotent teardown in correct order."""
    safe_call(sm.delete_endpoint, EndpointName=name)
    safe_call(sm.delete_endpoint_config, EndpointConfigName=name)
    safe_call(sm.delete_model, ModelName=name)

def wait_inservice(name: str):
    last = None
    start = time.time()
    while True:
        d = sm.describe_endpoint(EndpointName=name)
        st = d["EndpointStatus"]
        if st != last:
            print("Endpoint status:", st)
            last = st
        if st in ("InService", "Failed"):
            if st == "Failed":
                print("FailureReason:\n", d.get("FailureReason"))
            return st
        if (time.time() - start) > (AUTO_TIMEOUT_MIN * 60):
            print(f"Timeout waiting for InService (> {AUTO_TIMEOUT_MIN} min). Aborting.")
            return "TimedOut"
        time.sleep(10)